# Anhang: Vorversuche

| Abbildung | Label |
|---|---|
| Dense-Layer-Vergleich | `fig:dense_vorversuch` |
| Optimizer-Vergleich | `fig:optimizer_vergleich` |
| Fine-Tuning Layer-Vergleich | `fig:finetuning_layer_vergleich` |
| Feature Extraction vs. Fine-Tuning | `fig:finetuning_vergleich` |
| Augmentierungs-Tool-Vergleich | `aug_fig1_bar_val_acc.png` |

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '/Users/jan/PycharmProjects/DasWirdGut')

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers

from config import DATA, MODEL, PATHS
from training.model import build_model
from data.loader import load_train_val_datasets

NUM_CLASSES = len(DATA['classes'])

# Phase-1-Parameter — identisch zur finalen Pipeline
P1_LR      = 1e-2
P1_EPOCHS  = 50
P1_PATIENCE = 3

# Phase-2-Parameter — nur für Vorversuche in diesem Notebook
P2_LR       = 1e-5
P2_EPOCHS   = 10
P2_PATIENCE = 3

print('TF:', tf.__version__)
print('Klassen:', DATA['classes'])
print('Plots →', PATHS['plots'])

## Hilfsfunktionen

In [ ]:
def get_datasets():
    train_ds, val_ds = load_train_val_datasets()
    train_ds = train_ds.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
    val_ds   = val_ds.cache().prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds


def train_phase1(model, label=''):
    train_ds, val_ds = get_datasets()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=P1_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=P1_PATIENCE, restore_best_weights=True
    )
    print(f'\n── Phase 1: {label} ──')
    return model.fit(
        train_ds, validation_data=val_ds,
        epochs=P1_EPOCHS, callbacks=[cb], verbose=1
    )


def plot_two_metrics(histories: dict, title: str, filename: str):
    palette = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']
    styles  = ['-', '--', '-.', ':']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, ylabel in zip(
        axes,
        ['val_accuracy', 'val_loss'],
        ['Validation Accuracy', 'Validation Loss']
    ):
        for (lbl, h), color, ls in zip(histories.items(), palette, styles):
            ax.plot(h.history[metric], label=lbl, color=color, linestyle=ls)
        ax.set_title(ylabel)
        ax.set_xlabel('Epoche')
        ax.legend(fontsize=8)
        ax.grid(linestyle='--', alpha=0.4)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    out = PATHS['plots'] / filename
    plt.savefig(out, dpi=200, bbox_inches='tight')
    print(f'Gespeichert: {out}')
    plt.show()

---
## 1. `fig:dense_vorversuch` – Dense-Layer-Vergleich

In [ ]:
def build_model_with_dense(num_classes):
    base = tf.keras.applications.MobileNetV2(
        input_shape=tuple(DATA['img_size']) + (3,),
        include_top=False, weights='imagenet'
    )
    base.trainable = False
    inputs  = tf.keras.Input(shape=tuple(DATA['img_size']) + (3,))
    x       = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dropout(MODEL['dropout'])(x)
    x       = layers.Dense(128, activation='relu')(x)  # zusätzlicher Layer
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)


m_no_dense,   _ = build_model(NUM_CLASSES)
m_with_dense    = build_model_with_dense(NUM_CLASSES)

h_no_dense   = train_phase1(m_no_dense,   'Ohne Dense-Layer')
h_with_dense = train_phase1(m_with_dense, 'Mit Dense-Layer (128)')

plot_two_metrics(
    histories={
        'Ohne Dense-Layer':      h_no_dense,
        'Mit Dense-Layer (128)': h_with_dense,
    },
    title='Dense-Layer-Vergleich (Feature Extraction)',
    filename='dense_vorversuch.png'
)

---
## 2. `fig:optimizer_vergleich` – Adam vs. AdamW × Lernrate

In [ ]:
OPTIMIZER_CONFIGS = {
    'Adam  lr=1e-2': tf.keras.optimizers.Adam(learning_rate=1e-2),
    'Adam  lr=1e-3': tf.keras.optimizers.Adam(learning_rate=1e-3),
    'AdamW lr=1e-2': tf.keras.optimizers.AdamW(learning_rate=1e-2),
    'AdamW lr=1e-3': tf.keras.optimizers.AdamW(learning_rate=1e-3),
}

opt_histories = {}
for name, optimizer in OPTIMIZER_CONFIGS.items():
    train_ds, val_ds = get_datasets()
    model, _ = build_model(NUM_CLASSES)
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=P1_PATIENCE, restore_best_weights=True
    )
    print(f'\n── {name} ──')
    opt_histories[name] = model.fit(
        train_ds, validation_data=val_ds,
        epochs=P1_EPOCHS, callbacks=[cb], verbose=1
    )

plot_two_metrics(
    histories=opt_histories,
    title='Optimizer-Vergleich (Feature Extraction)',
    filename='optimizer_vergleich.png'
)

---
## 3. `fig:finetuning_layer_vergleich` – Frozen-Layer-Vergleich
> Phase 1 einmalig trainieren, Gewichte sichern, dann Fine-Tuning mit frozen=50/100/130.

In [ ]:
# Phase 1 — gemeinsamer Ausgangspunkt
model_p1, _ = build_model(NUM_CLASSES)
hist_p1     = train_phase1(model_p1, 'Basis für Fine-Tuning')
base_weights = model_p1.get_weights()

In [ ]:
def run_finetuning(frozen_until: int):
    train_ds, val_ds = get_datasets()
    model, backbone = build_model(NUM_CLASSES)
    model.set_weights(base_weights)

    backbone.trainable = True
    for layer in backbone.layers[:frozen_until]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=P2_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=P2_PATIENCE, restore_best_weights=True
    )
    print(f'\n── Fine-Tuning frozen={frozen_until} ──')
    return model.fit(
        train_ds, validation_data=val_ds,
        epochs=P2_EPOCHS, callbacks=[cb], verbose=1
    )


ft_layer_histories = {f'frozen={f}': run_finetuning(f) for f in [50, 100, 130]}

plot_two_metrics(
    histories=ft_layer_histories,
    title='Fine-Tuning: Vergleich eingefrorener Layer (50 / 100 / 130)',
    filename='finetuning_layer_vergleich.png'
)

---
## 4. `fig:finetuning_vergleich` – Feature Extraction vs. Fine-Tuning
> Nutzt `hist_p1` und `run_finetuning` aus Abschnitt 3 — Zellen von oben nach unten ausführen.

In [ ]:
hist_ft_100 = run_finetuning(100)

plot_two_metrics(
    histories={
        'Feature Extraction (Phase 1)':      hist_p1,
        'Fine-Tuning (Phase 2, frozen=100)': hist_ft_100,
    },
    title='Feature Extraction vs. Fine-Tuning',
    filename='finetuning_vergleich.png'
)

---
## 5. `aug_fig1_bar_val_acc.png` – Augmentierungs-Tool-Vergleich (korrigiert)

In [ ]:
AUG_CONFIGS = {
    'Keine Aug.\n(Referenz)': [],
    'Brightness':             [layers.RandomBrightness(0.1)],
    'Contrast':               [layers.RandomContrast(0.1)],
    'Flip':                   [layers.RandomFlip('horizontal_and_vertical')],
    'Rotation':               [layers.RandomRotation(0.1)],
    'Zoom':                   [layers.RandomZoom(height_factor=(-0.1, 0.0))],
    'Alle kombiniert':        [
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.1),
        layers.RandomBrightness(0.1),
        layers.RandomContrast(0.1),
        layers.RandomZoom(height_factor=(-0.1, 0.0)),
    ],
}

aug_results = {}
for config_name, aug_layers in AUG_CONFIGS.items():
    train_ds, val_ds = get_datasets()

    if aug_layers:
        augment  = tf.keras.Sequential(aug_layers, name='augmentation')
        train_ds = train_ds.map(
            lambda x, y: (augment(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        ).cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)

    model, _ = build_model(NUM_CLASSES)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=P1_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=P1_PATIENCE, restore_best_weights=True
    )
    print(f'\n── {config_name.strip()} ──')
    h = model.fit(
        train_ds, validation_data=val_ds,
        epochs=P1_EPOCHS, callbacks=[cb], verbose=1
    )
    aug_results[config_name] = max(h.history['val_accuracy'])
    print(f'  → Beste Val. Accuracy: {aug_results[config_name]:.4f}')

In [ ]:
ref_acc = aug_results['Keine Aug.\n(Referenz)']
labels  = list(aug_results.keys())
values  = list(aug_results.values())

colors = [
    '#888888' if name == 'Keine Aug.\n(Referenz)'
    else '#4CAF50' if acc >= ref_acc
    else '#E57373'
    for name, acc in aug_results.items()
]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
ax.axhline(y=ref_acc, color='#555555', linestyle='--',
           linewidth=1.2, label=f'Referenz ({ref_acc:.1%})')

for bar, val in zip(bars, values):
    delta     = val - ref_acc
    delta_str = f'+{delta:.1%}' if delta >= 0 else f'{delta:.1%}'
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{val:.1%}\n({delta_str})',
        ha='center', va='bottom', fontsize=8
    )

ax.set_ylabel('Validation Accuracy')
ax.set_title(
    'Validation Accuracy je Augmentierungskonfiguration\n'
    '(.keras-Modelle, Validierungsdatensatz)'
)
ax.set_ylim(0, min(1.15, max(values) + 0.15))
ax.legend(loc='lower right')
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.xticks(rotation=0, ha='center')
plt.tight_layout()

out = PATHS['plots'] / 'aug_fig1_bar_val_acc.png'
plt.savefig(out, dpi=200, bbox_inches='tight')
print(f'Gespeichert: {out}')
plt.show()

In [ ]:
# Augmentierungs-Tool-Visualisierung
# Zeigt Original + Ergebnis jedes Tools auf einem Beispielbild

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from config import DATA, PATHS
from tensorflow.keras import layers

# ── Beispielbild laden ────────────────────────────────────
# Passe den Pfad auf ein beliebiges Testbild an
SAMPLE_PATH = PATHS['dataset'] / 'A_PALLET' / 'A_mitte_016.jpg'

img = tf.keras.utils.load_img(str(SAMPLE_PATH), target_size=tuple(DATA['img_size']))
img_array = tf.keras.utils.img_to_array(img)          # float32, [0, 255]
img_tensor = tf.expand_dims(img_array, axis=0)        # (1, 224, 224, 3)

# ── Augmentierungs-Tools ──────────────────────────────────
AUG_TOOLS = {
    'Original':    None,
    'Brightness':  layers.RandomBrightness(0.1),
    'Contrast':    layers.RandomContrast(0.1),
    'Flip':        layers.RandomFlip('horizontal_and_vertical'),
    'Rotation':    layers.RandomRotation(0.1),
    'Zoom':        layers.RandomZoom(height_factor=(-0.1, 0.0)),
}

# ── Plot ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(AUG_TOOLS), figsize=(16, 3))

for ax, (name, tool) in zip(axes, AUG_TOOLS.items()):
    if tool is None:
        result = img_array
    else:
        result = tool(img_tensor, training=True)[0].numpy()

    ax.imshow(np.clip(result / 255.0, 0, 1))
    ax.set_title(name, fontsize=10)
    ax.axis('off')

plt.suptitle('Augmentierungs-Tools im Vergleich', fontsize=12, y=1.02)
plt.tight_layout()

out = PATHS['plots'] / 'aug_tool_visualisierung.png'
plt.savefig(out, dpi=200, bbox_inches='tight')
print(f'Gespeichert: {out}')
plt.show()

---
## Zusammenfassung

In [ ]:
files = [
    ('fig:dense_vorversuch',           'dense_vorversuch.png'),
    ('fig:optimizer_vergleich',        'optimizer_vergleich.png'),
    ('fig:finetuning_layer_vergleich', 'finetuning_layer_vergleich.png'),
    ('fig:finetuning_vergleich',       'finetuning_vergleich.png'),
    ('aug_fig1_bar_val_acc',           'aug_fig1_bar_val_acc.png'),
]
print('=== Erzeugte Anhang-Abbildungen ===')
for label, fname in files:
    path   = PATHS['plots'] / fname
    status = '✓' if path.exists() else '✗ FEHLT'
    print(f'  [{status}]  {label:42s} → {fname}')